# CÓDIGO 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")


def dataset_path(file_name):
    local_path = DATA_DIR / file_name
    platform_path = Path("/datasets") / file_name
    if local_path.exists():
        return local_path
    return platform_path


# Importar los archivos
company_trips = pd.read_csv(dataset_path("project_sql_result_01.csv"))
dropoff_trips = pd.read_csv(dataset_path("project_sql_result_04.csv"))

# Ver las primeras filas
print("Primeras filas de company_trips:")
print(company_trips.head())
print()
print("Primeras filas de dropoff_trips:")
print(dropoff_trips.head())
print()

# Estudiar la informaci?n general de los datasets
print("Informaci?n de company_trips:")
company_trips.info()
print()
print("Informaci?n de dropoff_trips:")
dropoff_trips.info()
print()

# Estad?sticas descriptivas
print("Estad?sticas descriptivas de company_trips:")
print(company_trips.describe())
print()
print("Estad?sticas descriptivas de dropoff_trips:")
print(dropoff_trips.describe())
print()

# Revisar valores ausentes
print("Valores ausentes en company_trips:")
print(company_trips.isna().sum())
print()
print("Valores ausentes en dropoff_trips:")
print(dropoff_trips.isna().sum())
print()

# Revisar duplicados
print("Duplicados en company_trips:", company_trips.duplicated().sum())
print("Duplicados en dropoff_trips:", dropoff_trips.duplicated().sum())

# Asegurar que los tipos de datos sean correctos
company_trips["trips_amount"] = company_trips["trips_amount"].astype(int)
dropoff_trips["average_trips"] = dropoff_trips["average_trips"].astype(float)

# Identificar los 10 principales barrios por finalizaci?n de viajes
top_10_dropoffs = dropoff_trips.sort_values(
    by="average_trips",
    ascending=False
).head(10)

print()
print("Top 10 barrios por promedio de finalizaciones de viajes:")
print(top_10_dropoffs)


# GráficO 1

In [ ]:
# Top 10 empresas de taxis por n?mero de viajes
top_10_companies = company_trips.sort_values(
    by="trips_amount",
    ascending=False
).head(10)

plt.figure(figsize=(12, 6))
plt.barh(
    top_10_companies["company_name"],
    top_10_companies["trips_amount"]
)
plt.gca().invert_yaxis()
plt.title("Top 10 empresas de taxis por n?mero de viajes")
plt.xlabel("N?mero de viajes")
plt.ylabel("Empresa de taxis")
plt.tight_layout()
plt.show()


# Gráfico 2

In [ ]:
# top 10 barrios por número promedio de finalizaciones
plt.figure(figsize=(10, 6))
plt.bar(
    top_10_dropoffs["dropoff_location_name"],
    top_10_dropoffs["average_trips"]
)
plt.title("Top 10 barrios por promedio de finalizaciones de viajes")
plt.xlabel("Barrio")
plt.ylabel("Promedio de viajes")
plt.xticks(rotation=45)
plt.show()

# Prueba de hipótesis

## Hip?tesis nula, H0:

La duraci?n promedio de los viajes desde Loop hasta O'Hare es igual en s?bados con buen clima y s?bados con mal clima.

## Hip?tesis alternativa, H1:

La duraci?n promedio de los viajes desde Loop hasta O'Hare cambia en s?bados con mal clima.


## Criterio usado

Se utiliz? una prueba t para dos muestras independientes, porque comparamos la duraci?n promedio de dos grupos distintos:

* viajes con clima Good;
* viajes con clima Bad.

Antes de ejecutar el t-test se aplic? la prueba de Levene para evaluar si las varianzas de ambos grupos pod?an considerarse iguales. La decisi?n sobre el par?metro `equal_var` se tom? con base en el p-value de Levene:

* si el p-value de Levene es mayor o igual que `alpha = 0.05`, se usa `equal_var=True`;
* si el p-value de Levene es menor que `alpha = 0.05`, se usa `equal_var=False`.

De esta forma, la elecci?n del tipo de t-test queda respaldada por una prueba estad?stica y no por la diferencia en los tama?os de las muestras.


In [ ]:
import pandas as pd
from scipy import stats

# Importar el archivo
rides = pd.read_csv(dataset_path("project_sql_result_07.csv"))

# Estudiar los datos
print("Primeras filas de rides:")
print(rides.head())
print()
print("Informaci?n de rides:")
rides.info()
print()
print("Estad?sticas descriptivas de rides:")
print(rides.describe())
print()
print("Distribuci?n de condiciones clim?ticas:")
print(rides["weather_conditions"].value_counts())
print()

# Asegurar tipos correctos
rides["start_ts"] = pd.to_datetime(rides["start_ts"])
rides["duration_seconds"] = rides["duration_seconds"].astype(float)

# Revisar y eliminar duplicados
print("Duplicados antes de limpiar:", rides.duplicated().sum())
rides = rides.drop_duplicates()
print("Filas tras eliminar duplicados:", len(rides))
print()

# Verificar que todos los registros sean s?bados
print("D?as de la semana presentes en rides:")
print(rides["start_ts"].dt.day_name().unique())
print()

# Identificar y eliminar viajes con duraci?n igual a 0
print("Viajes con duraci?n 0:", (rides["duration_seconds"] == 0).sum())
rides = rides[rides["duration_seconds"] > 0]
print("Filas finales:", len(rides))
print()

# Separar los viajes seg?n el clima despu?s de limpiar los datos
bad_weather = rides[rides["weather_conditions"] == "Bad"]["duration_seconds"]
good_weather = rides[rides["weather_conditions"] == "Good"]["duration_seconds"]

# Calcular estad?sticas descriptivas por grupo
bad_mean = bad_weather.mean()
good_mean = good_weather.mean()
difference = bad_mean - good_mean

print("Cantidad de viajes con mal clima:", len(bad_weather))
print("Cantidad de viajes con buen clima:", len(good_weather))
print(f"Duraci?n promedio con mal clima: {bad_mean:.2f} segundos")
print(f"Duraci?n promedio con buen clima: {good_mean:.2f} segundos")
print(f"Diferencia aproximada: {difference:.2f} segundos")
print(f"Diferencia aproximada en minutos: {difference / 60:.2f}")
print()

# Prueba de Levene para decidir si se asumen varianzas iguales
alpha = 0.05
levene_stat, levene_p = stats.levene(bad_weather, good_weather)
equal_var = levene_p >= alpha

print(f"Estad?stico de Levene: {levene_stat:.4f}")
print(f"Valor p de Levene: {levene_p:.4f}")
print("?Se asumen varianzas iguales?:", equal_var)
print()

# Prueba t para dos muestras independientes
results = stats.ttest_ind(
    bad_weather,
    good_weather,
    equal_var=equal_var
)

print(f"p-value del t-test: {results.pvalue:.12f}")

if results.pvalue < alpha:
    print("Rechazamos la hip?tesis nula")
else:
    print("No podemos rechazar la hip?tesis nula")


## Datos:

Despu?s de ejecutar la limpieza, los resultados principales son:

* registros originales en `rides`: 1068;
* duplicados eliminados: 197;
* registros con `duration_seconds = 0` eliminados: 6;
* registros finales despu?s de la limpieza: 865;
* viajes con buen clima: 717;
* viajes con mal clima: 148;
* duraci?n promedio con buen clima: aproximadamente 2049.26 segundos;
* duraci?n promedio con mal clima: aproximadamente 2409.23 segundos;
* diferencia aproximada: 359.97 segundos, es decir, cerca de 6 minutos.

El p-value de Levene y el p-value del t-test se generan en la salida del c?digo anterior para que no dependan de valores escritos manualmente.


## Conclusi?n de la hip?tesis

Con un nivel de significaci?n de `alpha = 0.05`, la decisi?n se toma comparando el p-value del t-test con ese umbral. Si el p-value es menor que 0.05, se rechaza la hip?tesis nula; si es mayor o igual que 0.05, no se puede rechazar.

Despu?s de limpiar los datos y aplicar la prueba de Levene, el contraste permite evaluar de forma m?s confiable si la duraci?n promedio de los viajes desde Loop hasta el Aeropuerto Internacional O'Hare cambia los s?bados con mal clima.

Con los datos limpios, los viajes con mal clima duran en promedio aproximadamente 360 segundos m?s que los viajes con buen clima, es decir, cerca de 6 minutos adicionales. Desde una perspectiva de negocio, esto indica que el clima es un factor relevante para Zuber: en s?bados con lluvia o tormenta, los viajes hacia O'Hare tienden a tardar m?s, por lo que la empresa deber?a considerar estas condiciones al estimar tiempos de llegada, asignar conductores y planificar la disponibilidad del servicio.


# ANÁLISIS 

### Análisis del gráfico de empresas de taxis

El gráfico muestra una concentración importante de viajes en un grupo reducido de compañías. Flash Cab lidera claramente el mercado con 19,558 viajes durante el 15 y 16 de noviembre de 2017, seguida por Taxi Affiliation Services con 11,422 viajes y Medallion Leasin con 10,367 viajes. Esta diferencia indica que Flash Cab tenía una presencia operativa considerablemente mayor que sus competidores directos en el periodo analizado.

También se observa que varias empresas, como Yellow Cab, Taxi Affiliation Service Yellow, Chicago Carriage Cab Corp, City Service y Sun Taxi, mantienen volúmenes relevantes, aunque por debajo del grupo líder. Sin embargo, después de las primeras compañías, la cantidad de viajes disminuye de forma marcada. Esto sugiere que el mercado de taxis en Chicago está parcialmente concentrado: unas pocas empresas realizan una proporción elevada de los viajes, mientras que muchas compañías pequeñas tienen una participación mucho menor.

Desde la perspectiva de Zuber, este patrón es importante porque permite identificar a los principales competidores. Flash Cab, Taxi Affiliation Services y Medallion Leasin deberían considerarse referentes clave para analizar cobertura, disponibilidad de vehículos, posicionamiento y capacidad operativa. Además, la presencia de muchas empresas con bajo volumen podría representar una oportunidad para Zuber si logra ofrecer un servicio más eficiente o diferenciado.

### Análisis del gráfico de los 10 principales barrios de destino

El segundo gráfico muestra los barrios con mayor promedio de viajes finalizados durante noviembre de 2017. Loop ocupa el primer lugar con un promedio aproximado de 10,727 viajes, seguido por River North con 9,524 y Streeterville con 6,665. Estos resultados evidencian una fuerte concentración de destinos en zonas céntricas y de alta actividad urbana.

La presencia de barrios como Loop, River North, Streeterville y West Loop sugiere que una gran parte de la demanda se relaciona con áreas comerciales, oficinas, hoteles, restaurantes, turismo y entretenimiento. Estos sectores suelen generar un flujo constante de pasajeros durante diferentes momentos del día, lo que los convierte en zonas estratégicas para empresas de transporte.

También destaca O’Hare, que aparece dentro de los cinco principales destinos. Esto es relevante porque los aeropuertos suelen concentrar viajes de mayor distancia y posiblemente de mayor valor económico. Para Zuber, esta información puede ser útil al diseñar estrategias de disponibilidad de conductores cerca de zonas de alta demanda y rutas frecuentes hacia el aeropuerto.

En conjunto, los datos muestran que la demanda de viajes no está distribuida de manera uniforme en toda la ciudad. Por el contrario, se concentra en zonas específicas con alta actividad económica, turística y de transporte. Para una empresa nueva como Zuber, esto implica que una estrategia inicial podría enfocarse en cubrir de manera eficiente los barrios con mayor volumen de finalizaciones, especialmente Loop, River North, Streeterville, West Loop y O’Hare.

## Conclusión general del análisis exploratorio

El análisis exploratorio revela dos patrones principales. Primero, el mercado de taxis presenta una estructura competitiva concentrada, donde pocas empresas dominan una gran parte de los viajes. Segundo, la demanda de destinos se concentra en barrios céntricos y zonas estratégicas como el aeropuerto O’Hare.

Para Zuber, estos hallazgos son relevantes porque permiten identificar tanto a los competidores más fuertes como las áreas geográficas con mayor demanda potencial. Una estrategia de entrada al mercado podría priorizar la disponibilidad de vehículos en zonas como Loop, River North y Streeterville, además de fortalecer la cobertura hacia y desde O’Hare. Esto permitiría competir en los segmentos con mayor volumen de pasajeros y mejorar la eficiencia operativa desde el inicio.